## Imports

In [ ]:
import torch
import cv2
import numpy as np
from art.attacks.evasion import (
    FastGradientMethod,
    CarliniL2Method,
    ProjectedGradientDescent,
    SquareAttack,
    Wasserstein,
    PixelAttack,
)
from art.estimators.classification import PyTorchClassifier
from pathlib import Path
from rosaid.utils.vis import subplot_images
import matplotlib.pyplot as plt

## Test Attack

In [ ]:
class ClfModel(torch.nn.Module):
    def __init__(self, model: torch.nn.Module):
        super(ClfModel, self).__init__()
        self.model = model

    def forward(self, x):
        return self.model(x)[0]


CLASSES = ["BENIGN", "DOS", "SUBFLOOD", "UNAUTHPUB", "UNAUTHSUB"]
clf_model = ClfModel(
    torch.load(
        r"rosaid\results\image_classification\ROSIDS23_blockcnn2d__nosampling_binary_20260102\best_model_full.pth",
        map_location=torch.device("cuda"),
    )
)
clf_model


In [ ]:
# num, parameters
nmodel = torch.load(
    r"rosaid\results\image_classification\ROSIDS23_blockcnn2d__nosampling_binary_20260111\best_model_full.pth",
    weights_only=False,
)
num_param = sum(p.numel() for p in nmodel.parameters() if p.requires_grad)
print(f"Number of trainable parameters in Millions: {num_param * 1e-6:.2f}M")

In [ ]:
clf_model.model.output_size

In [ ]:
fgsm = FastGradientMethod(
    estimator=PyTorchClassifier(
        model=clf_model,
        loss=torch.nn.CrossEntropyLoss(),
        clip_values=(0, 1),
        input_shape=(1, 100, 256),
        nb_classes=5,
        optimizer=torch.optim.Adam(clf_model.parameters(), lr=0.001),
    ),
    eps=0.1,
    batch_size=32,
    targeted=False,
)
cwm = CarliniL2Method(
    classifier=PyTorchClassifier(
        model=clf_model,
        loss=torch.nn.CrossEntropyLoss(),
        clip_values=(0, 1),
        input_shape=(1, 100, 256),
        nb_classes=5,
        optimizer=torch.optim.Adam(clf_model.parameters(), lr=0.001),
    ),
    max_iter=100,
    batch_size=32,
    targeted=False,
)
pgd = ProjectedGradientDescent(
    estimator=PyTorchClassifier(
        model=clf_model,
        loss=torch.nn.CrossEntropyLoss(),
        clip_values=(0, 1),
        input_shape=(1, 100, 256),
        nb_classes=5,
        optimizer=torch.optim.Adam(clf_model.parameters(), lr=0.001),
    ),
    eps=0.1,
    eps_step=0.1,
    max_iter=40,
    targeted=False,
    batch_size=32,
)
sq = SquareAttack(
    estimator=PyTorchClassifier(
        model=clf_model,
        loss=torch.nn.CrossEntropyLoss(),
        clip_values=(0, 1),
        input_shape=(1, 100, 256),
        nb_classes=5,
        optimizer=torch.optim.Adam(clf_model.parameters(), lr=0.001),
    ),
    batch_size=32,
    max_iter=10,
)
pxl = PixelAttack(
    classifier=PyTorchClassifier(
        model=clf_model,
        loss=torch.nn.CrossEntropyLoss(),
        clip_values=(0, 1),
        input_shape=(1, 100, 256),
        nb_classes=5,
        optimizer=torch.optim.Adam(clf_model.parameters(), lr=0.001),
    ),
    max_iter=10,
    targeted=False,
)
wstn = Wasserstein(
    estimator=PyTorchClassifier(
        model=clf_model,
        loss=torch.nn.CrossEntropyLoss(),
        clip_values=(0, 1),
        input_shape=(1, 100, 256),
        nb_classes=5,
        optimizer=torch.optim.Adam(clf_model.parameters(), lr=0.001),
    ),
    max_iter=10,
    targeted=False,
)

In [ ]:
img_root = Path(
    r"rosaid\data\rosids23_sessions\session_images"
)
imgs = list(img_root.rglob("*.png"))
# sort by file size in descending order
imgs = sorted(imgs, key=lambda x: x.stat().st_size, reverse=True)

In [ ]:
idx = 100
img_pth = imgs[idx]
img_lbl = img_pth.stem.split("_")[0]
print(f"Processing image: {img_pth}, label: {img_lbl}")
for atk in [fgsm, pgd, wstn, pxl]:
    print(f"Using attack: {atk.__class__.__name__}")
    img = cv2.imread(str(img_pth), cv2.IMREAD_GRAYSCALE)
    # pad or truncate to match (100,256)
    img = img[:100, :256]
    if img.shape[0] < 100:
        pad_height = 100 - img.shape[0]
        img = np.pad(img, ((0, pad_height), (0, 0)), mode="constant", constant_values=0)

    if img.shape[1] < 256:
        pad_width = 256 - img.shape[1]
        img = np.pad(img, ((0, 0), (0, pad_width)), mode="constant", constant_values=0)
    oimg = img.copy()
    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=0)
    img = np.expand_dims(img, axis=0)

    adv_out = atk.generate(x=img)
    img = oimg.copy()
    adv_img = adv_out.squeeze() * 255
    # mask adv img: zeroed out columns remain zeroed
    zeroed_cols = np.where(np.all(img == 0, axis=0))[0]
    masked_adv = adv_img.copy()
    masked_adv[:, zeroed_cols] = 0
    diff = np.abs(img - masked_adv)

    def model_output(x):
        # add batch and channel dimension if needed
        if len(x.shape) == 2:
            x = np.expand_dims(x, axis=0)
            x = np.expand_dims(x, axis=0)
        x = torch.tensor(x).float() / 255
        with torch.no_grad():
            outputs = clf_model(x.to("cuda"))
            _, predicted = torch.max(outputs, 1)
        return predicted.cpu().numpy()

    print(f"""Original prediction: {CLASSES[model_output(img)[0]]}, 
        Adversarial prediction: {CLASSES[model_output(adv_out)[0]]}, 
        Diff prediction: {CLASSES[model_output(diff)[0]]},
        Masked Adversarial prediction: {CLASSES[model_output(masked_adv)[0]]}""")
    subplot_images(
        [img, adv_img, masked_adv, diff],
        titles=["Original", "Adversarial", "Masked", "Diff"],
        cmap="gray",
        order=(1, -1),
    )


## Visualise F1 Scores

In [ ]:
from pathlib import Path
import pandas as pd

import seaborn as sns
from pathlib import Path
import matplotlib as mpl

# Set global font family and size
mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif", "serif"]
mpl.rcParams["font.size"] = 12  # Adjust per Springer guidelines
mpl.rcParams["axes.labelsize"] = 12
mpl.rcParams["axes.titlesize"] = 13
mpl.rcParams["xtick.labelsize"] = 11
mpl.rcParams["ytick.labelsize"] = 11
mpl.rcParams["legend.fontsize"] = 11

# Optional: PDF/LaTeX text rendering for enhanced output
mpl.rcParams["text.usetex"] = False


In [ ]:
eval_root = Path(
    r"rosaid\results\evaluation\adversarial_evaluation"
)
# its subfolder
stfpm_models = list(eval_root.iterdir())
print(stfpm_models)

In [ ]:
from sklearn.metrics import f1_score

final_results = []
for stfpm_model in stfpm_models:
    attack_types = list(stfpm_model.iterdir())
    for attack_type in attack_types:
        atk_name = (
            "MIM"
            if "mom" in attack_type.name.lower()
            else "PGD"
            if "proj" in attack_type.name.lower()
            else attack_type.name
        )
        rate = attack_type.name.split("_")[-1]
        print(f"Processing {stfpm_model.name} - {atk_name} at rate {rate}")
        if not (attack_type / "adversarial_classifier_results.csv").exists():
            print(
                f"Skipping {stfpm_model.name} - {atk_name} at rate {rate} as results file not found."
            )
            continue
        clf_df = pd.read_csv(attack_type / "adversarial_classifier_results.csv")

        orig_micro_f1 = f1_score(
            clf_df["true_binary_label"], clf_df["clf_pred_clean"], average="micro"
        )
        adv_micro_f1 = f1_score(
            clf_df["true_binary_label"], clf_df["clf_pred_adv"], average="micro"
        )
        adv_masked_micro_f1 = f1_score(
            clf_df["true_binary_label"], clf_df["clf_pred_masked_adv"], average="micro"
        )
        orig_macro_f1 = f1_score(
            clf_df["true_binary_label"], clf_df["clf_pred_clean"], average="macro"
        )
        adv_macro_f1 = f1_score(
            clf_df["true_binary_label"], clf_df["clf_pred_adv"], average="macro"
        )
        adv_masked_macro_f1 = f1_score(
            clf_df["true_binary_label"], clf_df["clf_pred_masked_adv"], average="macro"
        )
        stfpm_name = stfpm_model.name
        if "resnet" in stfpm_name.lower():
            stfpm_name = "ResNet18"
        final_results.append(
            {
                "STFPM Model": stfpm_model.name,
                "Attack": atk_name,
                "Rate": rate,
                "Original Micro F1": orig_micro_f1,
                "Adv Micro F1": adv_micro_f1,
                "Adv Masked Micro F1": adv_masked_micro_f1,
                "Original Macro F1": orig_macro_f1,
                "Adv Macro F1": adv_macro_f1,
                "Adv Masked Macro F1": adv_masked_macro_f1,
                "Original Accuracy": (
                    clf_df["true_binary_label"] == clf_df["clf_pred_clean"]
                ).mean(),
                "Adv Accuracy": (
                    clf_df["true_binary_label"] == clf_df["clf_pred_adv"]
                ).mean(),
                "Adv Masked Accuracy": (
                    clf_df["true_binary_label"] == clf_df["clf_pred_masked_adv"]
                ).mean(),
            }
        )
        # break
    # break

In [ ]:
full_result_df = pd.DataFrame(final_results)
full_result_df

### Adv Macro F1

In [ ]:
f1_col = "Adv Macro F1"  # "Adv Masked Macro F1"
for grp, result_df in full_result_df.groupby("STFPM Model"):
    unique_attacks = result_df["Attack"].unique()
    n_attacks = len(unique_attacks)

    fig, axes = plt.subplots(n_attacks, 1, figsize=(8, 2 * n_attacks))
    if n_attacks == 1:
        axes = [axes]

    for idx, attack_type in enumerate(unique_attacks):
        ax = axes[idx]

        # Filter rows for this attack type
        attack_rows = result_df[result_df["Attack"] == attack_type]

        # Bars: Clean + this attack's rates
        bars = [result_df["Original Macro F1"].values[0]]
        labels = ["Clean"]
        for _, row in attack_rows.iterrows():
            bars.append(row["Adv Macro F1"])
            labels.append(f"{row.Attack}\n({row.Rate})")

        # Barplot with palette good for academia
        sns.barplot(x=labels, y=bars, ax=ax, palette="crest")

        # annotate bars with values; values in center of bars
        for i, v in enumerate(bars):
            ax.text(
                i,
                v - 0.09,
                f"{v:.3f}",
                # color="white" if v < 0.5 else "black",
                fontweight="bold",
                ha="center",
                va="center",
            )

        # FIXED SAFE ANNOTATIONS - Always inside plot bounds
        ax.set_ylim(0, 1.02)  # Set ylim FIRST
        min_y, max_y = ax.get_ylim()
        y_range = max_y - min_y

        ax.set_ylabel("Macro F1 Score", fontweight="bold", fontsize=14)
        ax.set_title(f"Attack: {attack_type}", fontweight="bold", fontsize=14)
        ax.grid(axis="y", linestyle="--", alpha=0.7)
        # ax.tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.savefig(
        f"adversarial_f1_{grp.replace(' ', '_')}.pdf", dpi=300, bbox_inches="tight"
    )
    plt.show()

### Both in LIne

In [ ]:
f1_col_1 = "Adv Macro F1"
f1_col_2 = "Adv Masked Macro F1"


for grp, result_df in full_result_df.groupby("STFPM Model"):
    unique_attacks = result_df["Attack"].unique()

    fig, ax = plt.subplots(figsize=(5, 3))

    all_f1_vals = []

    for attack_type in unique_attacks:
        # Filter rows for this attack type
        attack_rows = result_df[result_df["Attack"] == attack_type]

        # Prepare data: Clean (0.00) + attack rates, with two F1 values per rate
        rates = [0.0]
        f1_1_vals = [result_df["Original Macro F1"].values[0]]
        f1_2_vals = [result_df["Original Macro F1"].values[0]]

        for _, row in attack_rows.iterrows():
            rates.append(float(row.Rate))
            f1_1_vals.append(row[f1_col_1])
            f1_2_vals.append(row[f1_col_2])

        # Sort by rate for proper line plotting
        sorted_data = sorted(zip(rates, f1_1_vals, f1_2_vals))
        rates, f1_1_vals, f1_2_vals = zip(*sorted_data)

        # Collect all F1 values for y-axis limits
        all_f1_vals.extend(f1_1_vals)
        all_f1_vals.extend(f1_2_vals)

        # Plot lines
        ax.plot(
            rates,
            f1_1_vals,
            marker="o",
            linewidth=2.5,
            markersize=8,
            label=f"{attack_type}",
        )
        ax.plot(
            rates,
            f1_2_vals,
            marker="s",
            linewidth=2.5,
            markersize=8,
            linestyle="--",
            label=f"{attack_type} (Masked)",
        )

    # Set y-axis limits: min - 0.05 to 1.0
    min_f1 = min(all_f1_vals)
    y_min = max(0, min_f1 - 0.05)
    ax.set_ylim(y_min, 1.05)

    # Formatting
    ax.set_xlabel("Attack Rate ($\epsilon$)", fontweight="bold", fontsize=14)
    ax.set_ylabel("Macro F1 Score", fontweight="bold", fontsize=14)
    # tick sizes
    ax.tick_params(axis="x", labelsize=12)
    ax.tick_params(axis="y", labelsize=12)
    # ax.set_title(f"Model: {grp}", fontweight="bold", fontsize=14)
    ax.legend(fontsize=12, loc="best")
    ax.grid(True, linestyle="--", alpha=0.7)

    plt.tight_layout()
    plt.savefig("adversarial_f1.pdf", dpi=300, bbox_inches="tight")
    plt.show()


### Adv Masked

In [ ]:
for grp, result_df in full_result_df.groupby("STFPM Model"):
    unique_attacks = result_df["Attack"].unique()
    n_attacks = len(unique_attacks)

    fig, axes = plt.subplots(n_attacks, 1, figsize=(8, 2 * n_attacks))
    if n_attacks == 1:
        axes = [axes]

    for idx, attack_type in enumerate(unique_attacks):
        ax = axes[idx]

        # Filter rows for this attack type
        attack_rows = result_df[result_df["Attack"] == attack_type]

        # Bars: Clean + this attack's rates
        bars = [result_df["Original Macro F1"].values[0]]
        labels = ["Clean"]
        for _, row in attack_rows.iterrows():
            bars.append(row["Adv Masked Macro F1"])
            labels.append(f"{row.Attack}\n({row.Rate})")

        # Barplot with palette good for academia
        sns.barplot(x=labels, y=bars, ax=ax, palette="crest")

        # annotate bars with values; values in center of bars
        for i, v in enumerate(bars):
            ax.text(
                i,
                v - 0.09,
                f"{v:.3f}",
                # color="white" if v < 0.5 else "black",
                fontweight="bold",
                ha="center",
                va="center",
            )

        # FIXED SAFE ANNOTATIONS - Always inside plot bounds
        ax.set_ylim(0, 1.02)  # Set ylim FIRST
        min_y, max_y = ax.get_ylim()
        y_range = max_y - min_y

        ax.set_ylabel("Macro F1 Score", fontweight="bold", fontsize=14)
        ax.set_title(f"Attack: {attack_type}", fontweight="bold", fontsize=14)
        ax.grid(axis="y", linestyle="--", alpha=0.7)
        # ax.tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.savefig(
        f"adversarial_masked_f1_{grp.replace(' ', '_')}.pdf",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()


### Anomaly Score

In [ ]:
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

feat_results = []
all_adv_scores = {}
baseline_clean_scores = None

# First pass: Collect baseline clean scores (first model, first attack)
first_model = next(iter(stfpm_models))
first_attack = next(iter(first_model.iterdir()))
baseline_feat_df = pd.read_csv(first_attack / "adversarial_stfpm_results.csv")
baseline_clean_scores = baseline_feat_df["clean_anomaly_score"]

baseline_benign_scores = baseline_feat_df.query("true_binary_label!='ATTACK'")[
    "adv_anomaly_score"
].values
baseline_atk_scores = baseline_feat_df.query("true_binary_label=='ATTACK'")[
    "adv_anomaly_score"
].values

adv_score_col = "masked_adv_anomaly_score"  # masked_adv_anomaly_score"
print(f"Baseline clean scores collected from {first_model.name}")

# Collect all adversarial scores
for stfpm_model in stfpm_models:
    stfpm_name = (
        "STFPM: DNP3 Teacher"
        if "dnp3" in stfpm_model.name.lower()
        else "STFPM: ROSIDS23 Teacher"
    )
    attack_types = list(stfpm_model.iterdir())
    for attack_type in attack_types:
        atk_name = (
            "MIM"
            if "mom" in attack_type.name.lower()
            else "PGD"
            if "proj" in attack_type.name.lower()
            else attack_type.name
        )
        rate = attack_type.name.split("_")[-1]
        print(f"Processing {stfpm_model.name} - {atk_name} at rate {rate}")

        feat_df = pd.read_csv(attack_type / "adversarial_stfpm_results.csv")

        adv_scores = feat_df[adv_score_col]

        non_atk_scores = feat_df.query("true_binary_label!='ATTACK'")[
            adv_score_col
        ].values
        atk_scores = feat_df.query("true_binary_label=='ATTACK'")[adv_score_col].values

        # Store adversarial scores with proper label
        label = f"{atk_name} ({rate})"
        all_adv_scores[label] = [non_atk_scores, atk_scores]

        # KS test vs baseline clean
        ks_result = ks_2samp(baseline_clean_scores, adv_scores)
        feat_results.append(
            {
                "model": stfpm_model.name,
                "attack": atk_name,
                "rate": rate,
                "ks_statistic": ks_result.statistic,
                "p_value": ks_result.pvalue,
            }
        )

        if ks_result.pvalue < 0.05:
            print(f"  → Significant difference (p={ks_result.pvalue:.2e})")

    # FINAL BOX PLOT: Clean vs All Attacks in 2,3 sub plot
    # clean in top left, rest filled with attacks
    fig, axes = plt.subplots(figsize=(15, 8), nrows=2, ncols=7, sharey=True)

    all_scores = [[baseline_benign_scores, baseline_atk_scores]] + [
        scores for scores in all_adv_scores.values()
    ]
    labels = ["Non-Adversarial"] + list(all_adv_scores.keys())

    all_scores = [[baseline_benign_scores, baseline_atk_scores]]
    labels = ["Non-Adversarial"]

    for idx, (label, scores) in enumerate(all_adv_scores.items()):
        if idx == 6:
            all_scores.append([baseline_benign_scores, baseline_atk_scores])
            labels.append("Non-Adversarial")
        all_scores.append(scores)
        labels.append(label)

    axid = -1
    for ax, score_set, label in zip(axes.flatten(), all_scores, labels):
        axid += 1
        # log scoreset

        box_data = [np.log10(score_set[0]), np.log10(score_set[1])]
        bp = ax.boxplot(
            box_data,
            labels=["BENIGN", "ATTACK"],
            patch_artist=True,
            boxprops=dict(facecolor="#ADD8E6", color="black"),
            medianprops=dict(color="red"),
            whiskerprops=dict(color="black"),
            capprops=dict(color="black"),
            flierprops=dict(markerfacecolor="black", marker="o", markersize=5),
        )

        bp["boxes"][0].set_alpha(0.8)
        # ax.set_yscale("log")
        ax.set_title(label, fontsize=15, fontweight="bold")
        # ax.set_xticks([])
        if axid % 7 == 0:
            ax.set_ylabel("Anomaly Score (Log10)", fontsize=14, fontweight="bold")
        ax.grid(True, linestyle="--", alpha=0.7)

        # increase y ticks font size
        ax.yaxis.set_tick_params(labelsize=13)

    plt.tight_layout()
    stfpm_name = stfpm_name.replace(":", "").replace(" ", "_")
    print(f"Written to {stfpm_name}.pdf")
    plt.savefig(f"{stfpm_name}_{adv_score_col}.pdf", dpi=300)
    plt.show()


### Anomaly Violin

In [ ]:
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.patches import Patch

feat_results = []
all_adv_scores = {}
baseline_clean_scores = None


# First pass: Collect baseline clean scores (first model, first attack)
first_model = next(iter(stfpm_models))
first_attack = next(iter(first_model.iterdir()))
baseline_feat_df = pd.read_csv(first_attack / "adversarial_stfpm_results.csv")

# remove duplicate rtows with image_name
ndf = baseline_feat_df.drop_duplicates(subset=["image_name"])
baseline_clean_scores = ndf["clean_anomaly_score"]


baseline_benign_scores = ndf.query("true_binary_label!='ATTACK'")[
    "adv_anomaly_score"
].values
baseline_atk_scores = ndf.query("true_binary_label=='ATTACK'")[
    "adv_anomaly_score"
].values

adv_score_col = "adv_anomaly_score"
print(f"Baseline clean scores collected from {first_model.name}")
attack_types = list(stfpm_model.iterdir())


# Collect all adversarial scores
for stfpm_model in stfpm_models:
    stfpm_name = (
        "STFPM: DNP3 Teacher"
        if "dnp3" in stfpm_model.name.lower()
        else "STFPM: ROSIDS23 Teacher"
    )

    # Group by attack type
    attacks_by_type = {}
    for attack_type in attack_types:
        atk_name = (
            "MIM"
            if "mom" in attack_type.name.lower()
            else "PGD"
            if "proj" in attack_type.name.lower()
            else attack_type.name
        )
        rate = attack_type.name.split("_")[-1]

        if atk_name not in attacks_by_type:
            attacks_by_type[atk_name] = []

        print(f"Processing {stfpm_model.name} - {atk_name} at rate {rate}")

        feat_df = pd.read_csv(attack_type / "adversarial_stfpm_results.csv")
        adv_scores = feat_df[adv_score_col]

        non_atk_scores = feat_df.query("true_binary_label!='ATTACK'")[
            adv_score_col
        ].values
        atk_scores = feat_df.query("true_binary_label=='ATTACK'")[adv_score_col].values

        attacks_by_type[atk_name].append(
            {
                "rate": rate,
                "benign": non_atk_scores,
                "attack": atk_scores,
            }
        )

        # KS test vs baseline clean
        ks_result = ks_2samp(baseline_clean_scores, adv_scores)
        feat_results.append(
            {
                "model": stfpm_model.name,
                "attack": atk_name,
                "rate": rate,
                "ks_statistic": ks_result.statistic,
                "p_value": ks_result.pvalue,
            }
        )

        if ks_result.pvalue < 0.05:
            print(f"  → Significant difference (p={ks_result.pvalue:.2e})")

    # GROUPED VIOLIN PLOT: n rows (one per attack type), columns are rates
    n_attack_types = len(attacks_by_type)
    fig, axes = plt.subplots(
        n_attack_types, 1, figsize=(12, 3 * n_attack_types), sharey=True
    )

    if n_attack_types == 1:
        axes = [axes]

    # Calculate percentile of first violin (baseline benign)
    pctl = 97
    percentile = np.percentile(np.log10(baseline_benign_scores), pctl)
    print(f"{pctl} Percentile of baseline benign scores (log10): {percentile:.4f}")

    for row_idx, (atk_name, attack_data) in enumerate(attacks_by_type.items()):
        ax = axes[row_idx]

        # Prepare data for this attack type
        plot_data = []
        group_labels = []
        positions = []
        pos = 0
        group_pos = []

        # Reduced gap between groups: changed from 1.2 to 0.8
        group_spacing = 0.8

        # Add baseline (rate 0.000)
        plot_data.extend(
            [np.log10(baseline_benign_scores), np.log10(baseline_atk_scores)]
        )
        group_labels.append("0.000")
        group_pos.append(pos + 0.2)
        positions.extend([pos, pos + 0.4])
        pos += group_spacing

        # Add rates for this attack type
        for rate_data in attack_data:
            rate = rate_data["rate"]
            plot_data.extend(
                [np.log10(rate_data["benign"]), np.log10(rate_data["attack"])]
            )
            group_labels.append(rate)
            group_pos.append(pos + 0.2)
            positions.extend([pos, pos + 0.4])
            pos += group_spacing

        # Create grouped violin plot
        parts = ax.violinplot(
            plot_data,
            positions=positions,
            widths=0.35,
            showmeans=False,
            showmedians=True,
            showextrema=True,
        )

        # Color violins: light blue for benign, light red for attack
        for i, pc in enumerate(parts["bodies"]):
            if i % 2 == 0:  # Benign
                pc.set_facecolor("#ADD8E6")
            else:  # Attack
                pc.set_facecolor("#FFB6C6")
            pc.set_alpha(0.7)
            pc.set_edgecolor("black")
            pc.set_linewidth(1.5)

        # Style median lines and other components
        parts["cmedians"].set_edgecolor("red")
        parts["cmedians"].set_linewidth(2)
        parts["cbars"].set_edgecolor("black")
        parts["cmaxes"].set_edgecolor("black")
        parts["cmins"].set_edgecolor("black")

        # Draw horizontal dotted red line at percentile across all rows
        # show its label too
        ax.axhline(
            y=percentile,
            color="red",
            linestyle=":",
            linewidth=4,
            alpha=0.7,
            label="Baseline Benign",
        )

        # Set x-ticks and labels
        ax.set_xticks(group_pos)
        ax.set_xticklabels(group_labels, fontsize=10)

        # Only show x-labels in bottom row
        if row_idx < n_attack_types - 1:
            ax.set_xticklabels([])

        # xtitle on bottom row
        if row_idx == n_attack_types - 1:
            ax.set_xlabel("Attack Rate ($\epsilon$)", fontsize=11, fontweight="bold")

        ax.set_ylabel("Anomaly Score (Log10)", fontsize=11, fontweight="bold")
        ax.set_title(f"Attack Type: {atk_name}", fontsize=12, fontweight="bold")
        ax.grid(True, axis="y", linestyle="--", alpha=0.7)
        ax.yaxis.set_tick_params(labelsize=10)

    # Add legend to best position automatically

    legend_elements = [
        Patch(facecolor="#ADD8E6", edgecolor="black", label="Benign"),
        Patch(facecolor="#FFB6C6", edgecolor="black", label="Attack"),
        mpl.lines.Line2D(
            [0],
            [0],
            color="red",
            lw=4,
            linestyle=":",
            label=f"Baseline Benign {pctl} Percentile",
        ),
    ]
    axes[0].legend(handles=legend_elements, loc="lower right", fontsize=11)

    plt.tight_layout()
    stfpm_name_clean = stfpm_name.replace(":", "").replace(" ", "_")
    print(f"Written to {stfpm_name_clean}_{adv_score_col}.pdf")
    plt.savefig(f"{stfpm_name_clean}_{adv_score_col}.pdf", dpi=300, bbox_inches="tight")
    plt.show()


### Baseline Anomaly Plot

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score


eval_root = Path(
    r"rosaid\results\evaluation\adversarial_evaluation"
)


baseline_stfpm_results = [
    eval_root / r"DNP3\momentumiterativemethod_eps_0.1\adversarial_stfpm_results.csv",
    eval_root
    / r"ROSIDS23\momentumiterativemethod_eps_0.1\adversarial_stfpm_results.csv",
    # eval_root
    # / r"ResNet18\momentumiterativemethod_eps_0.1\adversarial_stfpm_results.csv",
]


# Create one figure with single plot
fig, ax = plt.subplots(figsize=(11, 8))


colors_line = ["blue", "green", "red"]  # Different colors for threshold lines
violin_colors = [
    "#ADD8E6",
    "#FFE4B5",
    "#90EE90",
]  # Different colors for violins (light blue, light orange)
x_offset = 0
thresholds_info = []


threshold_results = {}


for color_idx, baseline_pth in enumerate(baseline_stfpm_results):
    baseline_pth = str(baseline_pth)
    print(f"Processing baseline STFPM results from: {baseline_pth}")
    baseline_feat_df = pd.read_csv(baseline_pth)

    teacher_name = (
        "DNP3"
        if "dnp3" in baseline_pth.lower()
        else "ROSIDS23"
        if "rosids23" in baseline_pth.lower()
        else "ResNet18"
    )

    # remove duplicate rows with image_name
    ndf = baseline_feat_df.drop_duplicates(subset=["image_name"])

    baseline_benign_scores = ndf.query("true_binary_label!='ATTACK'")[
        "clean_anomaly_score"
    ].values
    baseline_atk_scores = ndf.query("true_binary_label=='ATTACK'")[
        "clean_anomaly_score"
    ].values

    true_anomaly_label = np.array(
        [1 if lbl == "ATTACK" else 0 for lbl in ndf["true_binary_label"].values]
    )

    # now find the best threshold on benign scores using pctl
    best_f1 = 0
    best_pctl = 0
    best_acc = 0
    for pctl in range(50, 100):
        threshold = np.percentile(baseline_benign_scores, pctl)
        pred_labels = np.array(
            [1 if score >= threshold else 0 for score in ndf["clean_anomaly_score"]]
        )
        f1 = f1_score(true_anomaly_label, pred_labels, average="macro")
        if f1 > best_f1:
            best_f1 = f1
            best_pctl = pctl
            best_acc = (true_anomaly_label == pred_labels).mean()
    print(f"Best percentile threshold: {best_pctl} with F1: {best_f1:.4f}")

    # Calculate percentile of benign scores (variable)
    pctl = best_pctl
    percentile_val = np.percentile(baseline_benign_scores, pctl)
    percentile_val_log10 = np.log10(percentile_val)
    print(f"{pctl}th percentile of benign scores: {percentile_val:.4f}")
    print(f"{pctl}th percentile of benign scores (Log10): {percentile_val_log10:.4f}")

    threshold_results[teacher_name] = {
        "percentile": pctl,
        "threshold_value": percentile_val,
        "threshold_value_log10": percentile_val_log10,
        "best_f1": best_f1,
        "accuracy": best_acc,
    }
    # Calculate per-class accuracy
    threshold = np.percentile(baseline_benign_scores, pctl)
    pred_labels = np.array(
        [1 if score >= threshold else 0 for score in ndf["clean_anomaly_score"]]
    )

    benign_accuracy = np.sum(pred_labels[true_anomaly_label == 0] == 0) / np.sum(
        true_anomaly_label == 0
    )
    attack_accuracy = np.sum(pred_labels[true_anomaly_label == 1] == 1) / np.sum(
        true_anomaly_label == 1
    )

    print(
        f"Benign Accuracy: {benign_accuracy:.4f}, Attack Accuracy: {attack_accuracy:.4f}"
    )

    # Store threshold info for later annotation
    thresholds_info.append(
        {
            "x_pos": x_offset + 1.5,
            "y_pos": percentile_val_log10,
            "f1": best_f1,
            "log10": percentile_val_log10,
            "color": colors_line[color_idx],
            "violin_color": violin_colors[color_idx],
            "benign_acc": benign_accuracy,
            "attack_acc": attack_accuracy,
            "teacher_name": teacher_name,
            "benign_x": x_offset + 1,
            "attack_x": x_offset + 2,
            "x_min": x_offset + 0.7,  # Left boundary of group
            "x_max": x_offset + 2.3,  # Right boundary of group
            "pctl": pctl,
        }
    )

    # violin plot
    box_data = [np.log10(baseline_benign_scores), np.log10(baseline_atk_scores)]

    parts = ax.violinplot(
        box_data,
        positions=[x_offset + 1, x_offset + 2],
        widths=0.7,
        showmeans=False,
        showmedians=True,
        showextrema=True,
    )

    # Color violins with different color for each STFPM model
    for pc in parts["bodies"]:
        pc.set_facecolor(violin_colors[color_idx])
        pc.set_alpha(0.7)
        pc.set_edgecolor("black")
        pc.set_linewidth(1.5)

    # Style median and other components
    parts["cmedians"].set_edgecolor("red")
    parts["cmedians"].set_linewidth(2.5)
    parts["cbars"].set_edgecolor("black")
    parts["cmaxes"].set_edgecolor("black")
    parts["cmins"].set_edgecolor("black")

    # Draw percentile threshold line with different color - only within group bounds
    threshold_line = percentile_val_log10
    ax.plot(
        [x_offset + 0.5, x_offset + 2.5],
        [threshold_line, threshold_line],
        color=colors_line[color_idx],
        linestyle=":",
        linewidth=3.5,
        alpha=0.8,
        label=f"{teacher_name} ({pctl}th pctl)",
    )

    x_offset += 2.2  # Reduced spacing to bring groups closer


# Calculate average y position for accuracy and title texts
y_min, y_max = ax.get_ylim()
y_center = (y_min + y_max) / 2


# Add all annotations after plotting violins
for info in thresholds_info:
    # Add teacher name in a box at top (same y position for both groups)
    # ax.text(
    #     info["x_pos"],
    #     y_max * 1.3,
    #     info["teacher_name"],
    #     ha="center",
    #     va="top",
    #     fontsize=16,
    #     fontweight="bold",
    #     bbox=dict(
    #         boxstyle="round,pad=0.7",
    #         facecolor=info["violin_color"],
    #         edgecolor="black",
    #         linewidth=2.5,
    #         alpha=0.9,
    #     ),
    # )
    ax.text(
        info["x_pos"],
        -10,
        f"{info['teacher_name']}\nPercentile: {info['pctl']}\nThreshold: {info['log10']:.3f}\nF1: {info['f1']:.3f}",
        ha="center",
        va="center",
        fontsize=13,
        fontweight="bold",
        bbox=dict(
            boxstyle="round,pad=0.6",
            facecolor=info["violin_color"],
            edgecolor=info["color"],
            linewidth=2.5,
            alpha=0.95,
        ),
    )

    # Add accuracy annotations at same y position for both groups
    ax.text(
        info["benign_x"],
        y_center,
        f"Acc: {info['benign_acc']:.3f}",
        ha="center",
        va="center",
        fontsize=14,
        fontweight="bold",
        bbox=dict(
            boxstyle="round,pad=0.5",
            facecolor=info["violin_color"],
            edgecolor="blue",
            linewidth=2.5,
        ),
    )
    ax.text(
        info["attack_x"],
        y_center,
        f"Acc: {info['attack_acc']:.3f}",
        ha="center",
        va="center",
        fontsize=14,
        fontweight="bold",
        bbox=dict(
            boxstyle="round,pad=0.5",
            facecolor=info["violin_color"],
            edgecolor="red",
            linewidth=2.5,
        ),
    )

ax.set_ylabel("Anomaly Score (Log₁₀)", fontsize=18, fontweight="bold")
# ax.set_title("STFPM Adversarial Evaluation", fontsize=18, fontweight="bold", pad=20)
ax.grid(True, linestyle="--", alpha=0.6, linewidth=1.2)
ax.yaxis.set_tick_params(labelsize=15)
ax.xaxis.set_tick_params(labelsize=15)
ax.set_xticks([1, 2, 3.2, 4.2])
ax.set_xticklabels(
    ["BENIGN", "ATTACK", "BENIGN", "ATTACK"], fontsize=15, fontweight="bold"
)
# ax.legend(fontsize=12, loc="upper center", frameon=True, fancybox=True, shadow=True)


# Increase spine width for academic publishing
for spine in ax.spines.values():
    spine.set_linewidth(2)


# Remove margins
plt.subplots_adjust(left=0.1, right=0.95, top=0.92, bottom=0.1)


plt.savefig("baseline_anomaly_detection.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from pathlib import Path

eval_root = Path(
    r"rosaid\results\evaluation\adversarial_evaluation"
)

baseline_stfpm_results = [
    eval_root / r"DNP3\momentumiterativemethod_eps_0.1\adversarial_stfpm_results.csv",
    eval_root
    / r"ROSIDS23\momentumiterativemethod_eps_0.1\adversarial_stfpm_results.csv",
]

# Academic palette
model_colors = {
    "DNP3": {
        "line": "#0057B7",  # deep blue for threshold
        "fill": "#aec7e8",  # light blue for body
    },
    "ROSIDS23": {
        "line": "#A52A2A",  # brown/dark red for threshold
        "fill": "#ffbb78",  # light orange for body
    },
}

fig, ax = plt.subplots(figsize=(10, 7))

x_offset = 0
thresholds_info = []

for baseline_pth in baseline_stfpm_results:
    baseline_pth = str(baseline_pth)
    print(f"Processing: {baseline_pth}")
    baseline_feat_df = pd.read_csv(baseline_pth)

    teacher_name = "DNP3" if "dnp3" in baseline_pth.lower() else "ROSIDS23"
    colors_this = model_colors[teacher_name]
    line_color, violin_color = colors_this["line"], colors_this["fill"]

    ndf = baseline_feat_df.drop_duplicates(subset=["image_name"])
    benign_scores = ndf.query("true_binary_label!='ATTACK'")[
        "clean_anomaly_score"
    ].values
    attack_scores = ndf.query("true_binary_label=='ATTACK'")[
        "clean_anomaly_score"
    ].values
    true_labels = np.array(
        [1 if lbl == "ATTACK" else 0 for lbl in ndf["true_binary_label"].values]
    )

    best_f1, best_pctl = 0, 0
    for pctl in range(50, 100):
        threshold = np.percentile(benign_scores, pctl)
        preds = np.array(
            [1 if s >= threshold else 0 for s in ndf["clean_anomaly_score"]]
        )
        f1 = f1_score(true_labels, preds, average="macro")
        if f1 > best_f1:
            best_f1, best_pctl = f1, pctl

    pctl = best_pctl
    threshold = np.percentile(benign_scores, pctl)
    pctl_log10 = np.log10(threshold)

    preds = np.array([1 if s >= threshold else 0 for s in ndf["clean_anomaly_score"]])
    ben_acc = np.sum(preds[true_labels == 0] == 0) / np.sum(true_labels == 0)
    atk_acc = np.sum(preds[true_labels == 1] == 1) / np.sum(true_labels == 1)

    thresholds_info.append(
        {
            "x_pos": x_offset + 1.5,
            "log10": pctl_log10,
            "f1": best_f1,
            "line_color": line_color,
            "violin_color": violin_color,
            "benign_acc": ben_acc,
            "attack_acc": atk_acc,
            "teacher_name": teacher_name,
            "benign_x": x_offset + 1,
            "attack_x": x_offset + 2,
            "pctl": pctl,
        }
    )

    box_data = [np.log10(benign_scores), np.log10(attack_scores)]
    parts = ax.violinplot(box_data, positions=[x_offset + 1, x_offset + 2], widths=0.7)

    for pc in parts["bodies"]:
        pc.set_facecolor(violin_color)
        pc.set_alpha(0.7)
        pc.set_edgecolor("black")
        pc.set_linewidth(1.5)

    for part in ["cbars", "cmaxes", "cmins"]:
        if part in parts:
            parts[part].set_edgecolor("black")

    ax.plot(
        [x_offset + 0.5, x_offset + 2.5],
        [pctl_log10, pctl_log10],
        color=line_color,
        linestyle="--",
        linewidth=3.0,
        alpha=0.95,
        label=f"{teacher_name} threshold ({pctl}th pctl)",
    )

    x_offset += 2.2

y_center = np.mean(ax.get_ylim())
for info in thresholds_info:
    ax.text(
        info["x_pos"],
        -10,
        f"{info['teacher_name']}\nPercentile: {info['pctl']}\nThreshold: {info['log10']:.3f}\nF1: {info['f1']:.3f}",
        ha="center",
        va="center",
        fontsize=13,
        fontweight="bold",
        bbox=dict(
            boxstyle="round,pad=0.15",
            facecolor=info["violin_color"],
            edgecolor="black",
            linewidth=1.5,
        ),
    )

    for x_p, acc in [
        (info["benign_x"], info["benign_acc"]),
        (info["attack_x"], info["attack_acc"]),
    ]:
        ax.text(
            x_p,
            y_center,
            f"Acc: {acc:.3f}",
            ha="center",
            va="center",
            fontsize=13,
            fontweight="bold",
            bbox=dict(
                boxstyle="round,pad=0.5",
                facecolor=info["violin_color"],
                edgecolor="black",
                linewidth=1.8,
            ),
        )

ax.set_ylabel("Anomaly Score (Log₁₀)", fontsize=18, fontweight="bold")
ax.grid(True, linestyle="--", alpha=0.6)
ax.yaxis.set_tick_params(labelsize=15)
ax.set_xticks([1, 2, 3.2, 4.2])
ax.set_xticklabels(
    ["BENIGN", "ATTACK", "BENIGN", "ATTACK"], fontsize=15, fontweight="bold"
)

# CRITICAL: Tighten X-axis to remove left/right margins
ax.set_xlim(0.4, 4.8)

handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(
    by_label.values(), by_label.keys(), fontsize=12, frameon=True, loc="upper right"
)

for spine in ax.spines.values():
    spine.set_linewidth(2)

# Manual margin adjustment to absolute edges
plt.subplots_adjust(left=0.08, right=0.98, top=0.95, bottom=0.1)
plt.savefig(
    "baseline_anomaly_detection.pdf", dpi=300, bbox_inches="tight", pad_inches=0.02
)
plt.show()


### Line: Rate vs F1

In [ ]:
threshold_results


In [ ]:
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.patches import Patch

thresholds = threshold_results

# thresholds = {
#     "DNP3": {
#         "percentile": 88,
#         "threshold_value": 1.8505119027593723e-19,
#         "threshold_value_log10": -18.73270811711516,
#         "best_f1": 0.862697953575659,
#         "accuracy": 0.8682727079344821,
#     },
#     "ROSIDS23": {
#         "percentile": 69,
#         "threshold_value": 1.1024399467585079e-20,
#         "threshold_value_log10": -19.95764505855104,
#         "best_f1": 0.7572698061198747,
#         "accuracy": 0.7723888534354393,
#     },
# }


# First pass: Collect baseline clean scores (first model, first attack)
first_model = next(iter(stfpm_models))
first_attack = next(iter(first_model.iterdir()))
baseline_feat_df = pd.read_csv(first_attack / "adversarial_stfpm_results.csv")
baseline_clean_scores = baseline_feat_df["clean_anomaly_score"]


ndf = baseline_feat_df.drop_duplicates(subset=["image_name"])
baseline_clean_scores = ndf["clean_anomaly_score"]
baseline_benign_scores = ndf.query("true_binary_label!='ATTACK'")[
    "adv_anomaly_score"
].values
baseline_atk_scores = ndf.query("true_binary_label=='ATTACK'")[
    "adv_anomaly_score"
].values


print(f"Baseline clean scores collected from {first_model.name}")


final_results = {}


# Collect all adversarial scores
for stfpm_model in stfpm_models:
    if "res" in stfpm_model.name.lower():
        print(
            f"Skipping model: {stfpm_model.name} (ResNet18 not evaluated in this analysis)"
        )
        continue
    print(f"Evaluating model: {stfpm_model.name}")

    # Determine teacher name and get threshold
    is_dnp3 = "dnp3" in stfpm_model.name.lower()
    teacher_name = (
        "DNP3"
        if is_dnp3
        else "ROSIDS23"
        if "rosids23" in stfpm_model.name.lower()
        else "ResNet18"
        if "resnet18" in stfpm_model.name.lower()
        else "Unknown"
    )
    threshold_log10 = thresholds[teacher_name]["threshold_value_log10"]

    stfpm_name = (
        "DNP3 Teacher"
        if is_dnp3
        else "ROSIDS23 Teacher"
        if "rosids23" in stfpm_model.name.lower()
        else "ResNet18 Teacher"
    )
    results = {
        "0.00": [
            thresholds[teacher_name]["accuracy"],
            thresholds[teacher_name]["accuracy"],
        ]
    }

    attack_types = list(stfpm_model.iterdir())
    for attack_type in attack_types:
        atk_name = (
            "MIM"
            if "mom" in attack_type.name.lower()
            else "PGD"
            if "proj" in attack_type.name.lower()
            else attack_type.name
        )
        rate = attack_type.name.split("_")[-1]
        # print(f"Processing {stfpm_model.name} - {atk_name} at rate {rate}")

        feat_df = pd.read_csv(attack_type / "adversarial_stfpm_results.csv")

        scores = []

        for adv_score_col in ["adv_anomaly_score", "masked_adv_anomaly_score"]:
            adv_scores = feat_df[adv_score_col]
            # Convert to log10
            adv_scores_log10 = np.log10(adv_scores)
            # everything is 1
            true_label = np.ones(len(adv_scores_log10))
            pred_labels = np.array(
                [1 if score >= threshold_log10 else 0 for score in adv_scores_log10]
            )
            accuracy = (true_label == pred_labels).mean()

            scores.append(accuracy)
        results[rate] = scores
    final_results[stfpm_name] = results


# Generate pandas table
table_data = []

for stfpm_name, results in final_results.items():
    for rate, accuracies in results.items():
        table_data.append(
            {
                "Model": stfpm_name,
                "Perturbation Rate": rate,
                "Adversarial Accuracy": accuracies[0],
                "Masked Adversarial Accuracy": accuracies[1],
            }
        )

df_results = pd.DataFrame(table_data)

# Display the table
print("\n" + "=" * 80)
print("ADVERSARIAL EVALUATION RESULTS")
print("=" * 80)
print(df_results.to_string(index=False))
print("=" * 80)

# Optional: Save to CSV
# df_results.to_csv("adversarial_evaluation_results.csv", index=False)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract unique models and rates
models = df_results["Model"].unique()
rates = df_results["Perturbation Rate"].unique()

# Convert rates to float for proper sorting
rate_values = sorted([float(r) for r in rates])
rate_labels = [str(r) for r in rate_values]

# Create figure
fig, ax = plt.subplots(figsize=(5, 4))

colors = [
    "#0066CC",
    "#FF6600",
    "#FF0000",
]  # Sharp blue for DNP3, sharp orange for ROSIDS23
line_styles = ["-", ":"]  # Solid for adversarial, dotted for masked
linewidth = 3.5
marker_size = 9
# Plot lines for each model
for model_idx, model in enumerate(models):
    model_data = df_results[df_results["Model"] == model]

    # Sort by rate
    model_data_sorted = model_data.copy()
    model_data_sorted["Rate_float"] = model_data_sorted["Perturbation Rate"].astype(
        float
    )
    model_data_sorted = model_data_sorted.sort_values("Rate_float")

    # Plot Adversarial Accuracy
    ax.plot(
        model_data_sorted["Perturbation Rate"],
        model_data_sorted["Adversarial Accuracy"],
        marker="o",
        markersize=marker_size,
        linewidth=linewidth,
        label=f"{model} (Adv.)",
        color=colors[model_idx],
        linestyle="-",
        alpha=0.85,
    )
    linewidth -= 0.5  # Decrease linewidth for masked line
    marker_size -= 1.5  # Decrease marker size for masked line

    # Plot Masked Adversarial Accuracy
    ax.plot(
        model_data_sorted["Perturbation Rate"],
        model_data_sorted["Masked Adversarial Accuracy"],
        marker="s",
        markersize=marker_size,
        linewidth=linewidth,
        label=f"{model} (Masked Adv.)",
        color=colors[model_idx],
        linestyle=":",
        alpha=0.75,
    )

    linewidth -= 0.5  # Decrease linewidth for masked line
    marker_size -= 1.5  # Decrease marker size for masked line

ax.set_xlabel("Attack Rate", fontsize=18, fontweight="bold")
ax.set_ylabel("Accuracy", fontsize=18, fontweight="bold")
# ax.set_title(
#     "Adversarial Robustness Evaluation", fontsize=20, fontweight="bold", pad=20
# )
ax.grid(True, alpha=0.5, linestyle="--", linewidth=1.3)
ax.tick_params(axis="both", labelsize=15)
ax.legend(
    fontsize=14, loc="best", frameon=True, fancybox=True, shadow=True, edgecolor="black"
)
ax.set_ylim([0, 1.05])

# Increase spine width
for spine in ax.spines.values():
    spine.set_linewidth(2.5)

plt.tight_layout()
plt.savefig("adversarial_robustness.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

models = df_results["Model"].unique()  # 2 models
rates = sorted(df_results["Perturbation Rate"].astype(float).unique())
rate_labels = [str(r) for r in rates]
n_rates = len(rates)

bar_width = 0.35  # Increased for grouped clarity
x = np.arange(n_rates)

fig, ax = plt.subplots(figsize=(6, 4.5))  # Slightly wider

# New color palette - distinct for each model-type combination
colors = {
    "Adv": ["#1f77b4", "#ff7f0e"],  # Blue, Orange
    "Masked": ["#2ca02c", "#d62728"],  # Green, Red
}

for m_idx, model in enumerate(models):
    model_data = df_results[df_results["Model"] == model].copy()
    model_data["Rate_float"] = model_data["Perturbation Rate"].astype(float)
    model_data = model_data.sort_values("Rate_float")

    # Grouped positioning: Adv left, Masked right
    adv_offset = (m_idx - 0.5) * bar_width
    masked_offset = (m_idx - 0.5) * bar_width + bar_width / 2

    # Adv bars (left position)
    ax.bar(
        x + adv_offset,
        model_data["Adversarial Accuracy"],
        width=bar_width / 2,
        color=colors["Adv"][m_idx],
        edgecolor="black",
        linewidth=1.2,
        label=f"{model} (Adv.)",
    )

    # Masked Adv bars (right position)
    ax.bar(
        x + masked_offset,
        model_data["Masked Adversarial Accuracy"],
        width=bar_width / 2,
        color=colors["Masked"][m_idx],
        edgecolor="black",
        linewidth=1.2,
        label=f"{model} (Masked Adv.)",
    )

ax.set_xticks(x)
ax.set_xticklabels(rate_labels, fontsize=12)
ax.set_xlabel("Attack Rate", fontsize=16, fontweight="bold")
ax.set_ylabel("Accuracy", fontsize=16, fontweight="bold")
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.4, linestyle="--", linewidth=1)
ax.tick_params(axis="both", labelsize=12)

# Clean legend
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(
    by_label.values(),
    by_label.keys(),
    fontsize=13,
    frameon=True,
    fancybox=True,
    edgecolor="black",
    loc="lower left",
    # bbox_to_anchor=(0.5, -0.15),
    ncol=1,
)

plt.tight_layout()
plt.savefig("adversarial_robustness.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

models = df_results["Model"].unique()  # 2 models
rates = sorted(df_results["Perturbation Rate"].astype(float).unique())
rate_labels = [str(r) for r in rates]
n_rates = len(rates)

bar_width = 0.35  # Increased for grouped clarity
x = np.arange(n_rates)

fig, ax = plt.subplots(figsize=(6, 4.5))  # Slightly wider

# New color palette - distinct for each model-type combination
colors = {
    "Adv": ["#1f77b4", "#ff7f0e"],  # Blue, Orange
    "Masked": ["#2ca02c", "#d62728"],  # Green, Red
}

for m_idx, model in enumerate(models):
    model_data = df_results[df_results["Model"] == model].copy()
    model_data["Rate_float"] = model_data["Perturbation Rate"].astype(float)
    model_data = model_data.sort_values("Rate_float")

    # Grouped positioning: Adv left, Masked right
    adv_offset = (m_idx - 0.5) * bar_width
    masked_offset = (m_idx - 0.5) * bar_width + bar_width / 2

    # Adv bars (left position)
    ax.bar(
        x + adv_offset,
        model_data["Adversarial Accuracy"],
        width=bar_width / 2,
        color=colors["Adv"][m_idx],
        edgecolor="black",
        linewidth=1.2,
        label=f"{model} (Adv.)",
    )

    # Masked Adv bars (right position)
    ax.bar(
        x + masked_offset,
        model_data["Masked Adversarial Accuracy"],
        width=bar_width / 2,
        color=colors["Masked"][m_idx],
        edgecolor="black",
        linewidth=1.2,
        label=f"{model} (Masked Adv.)",
    )

# ========== ADD AVERAGE LABELS ABOVE EACH GROUP ==========
for i in range(n_rates):
    # Group 1: bars 0,1 (Model1 Adv + Model1 Masked)
    bars_group1 = ax.patches[i::n_rates][:2]
    avg1 = np.mean([bar.get_height() for bar in bars_group1])

    # Group 2: bars 2,3 (Model2 Adv + Model2 Masked)
    bars_group2 = ax.patches[i::n_rates][2:]
    avg2 = np.mean([bar.get_height() for bar in bars_group2])

    # Position above each subgroup center (no box, safe margins)
    ax.text(
        x[i] - bar_width / 4,
        min(avg1 + 0.01, 1.02),
        f"{avg1:.2f}",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
    )
    ax.text(
        x[i] + bar_width / 1.19,
        min(avg2 + 0.01, 1.02),
        f"{avg2:.2f}",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
    )
ax.set_xticks(x)
ax.set_xticklabels(rate_labels, fontsize=12)
ax.set_xlabel("Attack Rate", fontsize=16, fontweight="bold")
ax.set_ylabel("Accuracy", fontsize=16, fontweight="bold")
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.4, linestyle="--", linewidth=1)
ax.tick_params(axis="both", labelsize=12)

# Clean legend
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(
    by_label.values(),
    by_label.keys(),
    fontsize=13,
    frameon=True,
    fancybox=True,
    edgecolor="black",
    loc="lower left",
    ncol=1,
)

plt.tight_layout()
plt.savefig("adversarial_robustness.pdf", dpi=300, bbox_inches="tight")
plt.show()


### Anomaly Detection with Threshold

In [ ]:
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.patches import Patch


# First pass: Collect baseline clean scores (first model, first attack)
first_model = next(iter(stfpm_models))
first_attack = next(iter(first_model.iterdir()))
baseline_feat_df = pd.read_csv(first_attack / "adversarial_stfpm_results.csv")
baseline_clean_scores = baseline_feat_df["clean_anomaly_score"]

ndf = baseline_feat_df.drop_duplicates(subset=["image_name"])
baseline_clean_scores = ndf["clean_anomaly_score"]
baseline_benign_scores = ndf.query("true_binary_label!='ATTACK'")[
    "adv_anomaly_score"
].values
baseline_atk_scores = ndf.query("true_binary_label=='ATTACK'")[
    "adv_anomaly_score"
].values

print(f"Baseline clean scores collected from {first_model.name}")

pctl = 97
threshold = np.percentile(np.log10(baseline_benign_scores), pctl)
print(f"{pctl} Percentile of baseline benign scores (raw): {threshold:.4f}")

final_results = {}

# Collect all adversarial scores
for stfpm_model in stfpm_models:
    print(f"Evaluating model: {stfpm_model.name}")
    all_adv_scores["0.000"] = [baseline_benign_scores, baseline_atk_scores]
    stfpm_name = (
        "STFPM: DNP3 Teacher"
        if "dnp3" in stfpm_model.name.lower()
        else "STFPM: ROSIDS23 Teacher"
    )
    results = {
        "0.00": [
            (baseline_benign_scores, baseline_atk_scores),
            (baseline_benign_scores, baseline_atk_scores),
        ]
    }

    attack_types = list(stfpm_model.iterdir())
    for attack_type in attack_types:
        atk_name = (
            "MIM"
            if "mom" in attack_type.name.lower()
            else "PGD"
            if "proj" in attack_type.name.lower()
            else attack_type.name
        )
        rate = attack_type.name.split("_")[-1]
        # print(f"Processing {stfpm_model.name} - {atk_name} at rate {rate}")

        feat_df = pd.read_csv(attack_type / "adversarial_stfpm_results.csv")

        scores = []

        for adv_score_col in ["adv_anomaly_score", "masked_adv_anomaly_score"]:
            adv_scores = feat_df[adv_score_col]

            non_atk_scores = feat_df.query("true_binary_label!='ATTACK'")[
                adv_score_col
            ].values
            atk_scores = feat_df.query("true_binary_label=='ATTACK'")[
                adv_score_col
            ].values
            scores.append((non_atk_scores, atk_scores))
        results[f"{atk_name} ({rate})"] = scores
    final_results[stfpm_name] = results

In [ ]:
from sklearn.metrics import f1_score


for model_name, result_dict in final_results.items():
    print(f"F1 Scores for model: {model_name}")
    for label, value in result_dict.items():
        benign_scores, atk_scores = value[0]
        masked_benign_scores, masked_atk_scores = value[1]
        # log 10 scores
        benign_scores = np.log10(benign_scores)
        atk_scores = np.log10(atk_scores)
        masked_benign_scores = np.log10(masked_benign_scores)
        masked_atk_scores = np.log10(masked_atk_scores)

        if label == "0.00":
            labels.append("0.00")
            non_masked_true_labels = np.array(
                [0] * len(benign_scores) + [1] * len(atk_scores)
            )
            non_masked_pred_labels = np.array(
                [1 if score > threshold else 0 for score in benign_scores]
                + [1 if score > threshold else 0 for score in atk_scores]
            )
            masked_true_labels = np.array(
                [0] * len(masked_benign_scores) + [1] * len(masked_atk_scores)
            )
            masked_pred_labels = np.array(
                [1 if score > threshold else 0 for score in masked_benign_scores]
                + [1 if score > threshold else 0 for score in masked_atk_scores]
            )
        else:
            labels.append(label)
            # everything is attack
            non_masked_true_labels = np.array(
                [1] * len(benign_scores) + [1] * len(atk_scores)
            )
            non_masked_pred_labels = np.array(
                [1 if score > threshold else 0 for score in benign_scores]
                + [1 if score > threshold else 0 for score in atk_scores]
            )
            masked_true_labels = np.array(
                [1] * len(masked_benign_scores) + [1] * len(masked_atk_scores)
            )
            masked_pred_labels = np.array(
                [1 if score > threshold else 0 for score in masked_benign_scores]
                + [1 if score > threshold else 0 for score in masked_atk_scores]
            )
        non_masked_f1 = f1_score(
            non_masked_true_labels, non_masked_pred_labels, average="macro"
        )
        masked_f1 = f1_score(masked_true_labels, masked_pred_labels, average="macro")
        print(
            f"{label} - Non-masked F1: {non_masked_f1:.4f}, Masked F1: {masked_f1:.4f}"
        )


In [ ]:
# Plot line plots with subplots
n_models = len(final_results)
fig, axes = plt.subplots(1, n_models, figsize=(8, 4), sharey=True)

if n_models == 1:
    axes = [axes]

all_f1_vals = []

for col_idx, (model_name, result_dict) in enumerate(final_results.items()):
    ax = axes[col_idx]
    print(f"F1 Scores for model: {model_name}")

    # Organize by attack type and rate
    attack_data = {}

    for label, value in result_dict.items():
        benign_scores, atk_scores = value[0]
        masked_benign_scores, masked_atk_scores = value[1]

        # log 10 scores
        benign_scores = np.log10(benign_scores)
        atk_scores = np.log10(atk_scores)
        masked_benign_scores = np.log10(masked_benign_scores)
        masked_atk_scores = np.log10(masked_atk_scores)

        if label == "0.00":
            rate_val = 0.0
            non_masked_true_labels = np.array(
                [0] * len(benign_scores) + [1] * len(atk_scores)
            )
            non_masked_pred_labels = np.array(
                [1 if score > threshold else 0 for score in benign_scores]
                + [1 if score > threshold else 0 for score in atk_scores]
            )
            masked_true_labels = np.array(
                [0] * len(masked_benign_scores) + [1] * len(masked_atk_scores)
            )
            masked_pred_labels = np.array(
                [1 if score > threshold else 0 for score in masked_benign_scores]
                + [1 if score > threshold else 0 for score in masked_atk_scores]
            )
        else:
            # Extract rate from label (e.g., "MIM (0.1)" -> 0.1)
            rate_val = float(label.split("(")[1].rstrip(")"))
            if rate_val == 0.001:
                continue
            # everything is attack
            non_masked_true_labels = np.array(
                [1] * len(benign_scores) + [1] * len(atk_scores)
            )
            non_masked_pred_labels = np.array(
                [1 if score > threshold else 0 for score in benign_scores]
                + [1 if score > threshold else 0 for score in atk_scores]
            )
            masked_true_labels = np.array(
                [1] * len(masked_benign_scores) + [1] * len(masked_atk_scores)
            )
            masked_pred_labels = np.array(
                [1 if score > threshold else 0 for score in masked_benign_scores]
                + [1 if score > threshold else 0 for score in masked_atk_scores]
            )

        non_masked_f1 = f1_score(
            non_masked_true_labels, non_masked_pred_labels, average="macro"
        )
        masked_f1 = f1_score(masked_true_labels, masked_pred_labels, average="macro")

        print(
            f"{label} - Non-masked F1: {non_masked_f1:.4f}, Masked F1: {masked_f1:.4f}"
        )

        # Extract attack type
        if label == "0.00":
            attack_type = "Clean"
        else:
            attack_type = label.split("(")[0].strip()

        if attack_type not in attack_data:
            attack_data[attack_type] = {"rates": [], "non_masked": [], "masked": []}

        attack_data[attack_type]["rates"].append(rate_val)
        attack_data[attack_type]["non_masked"].append(non_masked_f1)
        attack_data[attack_type]["masked"].append(masked_f1)

    # Plot lines for each attack type
    for attack_type, data in attack_data.items():
        # Sort by rates
        sorted_indices = np.argsort(data["rates"])
        rates_sorted = [data["rates"][i] for i in sorted_indices]
        non_masked_sorted = [data["non_masked"][i] for i in sorted_indices]
        masked_sorted = [data["masked"][i] for i in sorted_indices]

        all_f1_vals.extend(non_masked_sorted)
        all_f1_vals.extend(masked_sorted)

        # Plot lines with alpha for transparency
        ax.plot(
            rates_sorted,
            non_masked_sorted,
            marker="o",
            linewidth=4,
            markersize=12,
            alpha=0.8,
            label=f"{attack_type}",
        )
        ax.plot(
            rates_sorted,
            masked_sorted,
            marker="s",
            linewidth=4,
            markersize=12,
            linestyle="--",
            alpha=0.8,
            label=f"{attack_type} (Masked)",
        )

    # Formatting per subplot
    ax.set_xlabel("Attack Rate ($\epsilon$)", fontweight="bold", fontsize=16)
    if col_idx == 0:
        ax.set_ylabel("Macro F1 Score", fontweight="bold", fontsize=16)
    ax.set_title(model_name.replace("STFPM: ", ""), fontweight="bold", fontsize=16)
    ax.tick_params(axis="x", labelsize=14)
    ax.tick_params(axis="y", labelsize=14)
    ax.legend(fontsize=12, loc="best")
    ax.grid(True, linestyle="--", alpha=0.7, linewidth=1.5)

# Set y-axis limits
min_f1 = min(all_f1_vals)
y_min = max(0, min_f1 - 0.05)
axes[0].set_ylim(y_min, 1.05)
# plt.subplots_adjust(left=0.05, right=0.92, wspace=0.25, hspace=0.3)

plt.tight_layout()
plt.savefig("adv_detection_comparison.pdf", dpi=300, bbox_inches="tight")
plt.show()


## Images

In [ ]:
import cv2

rates = [0.01, 0.5]
label = "DOS"
for atk_type in ["mom", "proj"]:
    adv_images = []
    masked_adv_images = []
    adv_amaps = []
    masked_adv_amaps = []
    attack_labels = []
    for atk_name in Path(
        r"rosaid\results\evaluation\adversarial_evaluation\DNP3"
    ).iterdir():
        if atk_type.lower() not in atk_name.name.lower():
            continue
        rate = float(atk_name.name.split("_")[-1])
        if rate not in rates:
            continue

        attack_name = (
            "MIM"
            if "mom" in atk_name.name.lower()
            else "PGD"
            if "proj" in atk_name.name.lower()
            else atk_name.name
        )
        print(f"Processing attack: {attack_name} at rate {rate}")

        benign_img = cv2.imread(
            str(atk_name / f"{label}_clean.png"), cv2.IMREAD_GRAYSCALE
        )
        adv_img = cv2.imread(str(atk_name / f"{label}_adv.png"), cv2.IMREAD_GRAYSCALE)
        masked_adv = cv2.imread(
            str(atk_name / f"{label}_masked_adv.png"), cv2.IMREAD_GRAYSCALE
        )

        data = np.load(atk_name / f"{label}_anomaly_maps.npz")
        # 'clean_amap', 'adv_amap', 'masked_amap'
        benign_amap = data["clean_amap"]
        adv_amap = data["adv_amap"]
        masked_adv_amap = data["masked_amap"]

        # log10 all
        benign_amap = np.log10(benign_amap)
        adv_amap = np.log10(adv_amap)
        masked_adv_amap = np.log10(masked_adv_amap)

        # benign_amap = cv2.imread(
        #     str(atk_name / "DOS_clean_amap.png"), cv2.IMREAD_GRAYSCALE
        # )
        # adv_amap = cv2.imread(str(atk_name / "DOS_adv_amap.png"), cv2.IMREAD_GRAYSCALE)
        # masked_adv_amap = cv2.imread(
        #     str(atk_name / "DOS_masked_adv_amap.png"), cv2.IMREAD_GRAYSCALE
        # )

        adv_images.append(adv_img)
        masked_adv_images.append(masked_adv)
        attack_labels.append(f"{attack_name} ({rate})")

        adv_amaps.append(adv_amap)
        masked_adv_amaps.append(masked_adv_amap)

    # plot 2, N sub plot
    ncols = len(rates) + 1
    nrows = 4
    fig, axes = plt.subplots(nrows, ncols, figsize=(10, 6))  # Wider fig
    aspect = 8
    for rid, row in enumerate(axes):
        for cid, ax in enumerate(row):
            ax.set_xticks([])
            ax.set_yticks([])

            if cid == 0:  # Benign column
                if rid == 0:
                    ax.imshow(benign_img, cmap="gray")
                    ax.set_title(label, fontweight="bold")
                elif rid == 1:
                    anomaly_score = benign_amap.max()
                    im = ax.imshow(benign_amap, cmap="jet", interpolation="nearest")
                    fig.colorbar(im, ax=ax, shrink=0.6, aspect=aspect, pad=0.02)
                    ax.set_ylabel("A. Map", fontweight="bold")
                    ax.set_title(f"Log10 Score: {anomaly_score:.2f}", fontweight="bold")
                elif rid == 2:
                    ax.imshow(benign_img, cmap="gray")
                    ax.set_title(label, fontweight="bold")
                else:  # rid==3
                    anomaly_score = benign_amap.max()
                    im = ax.imshow(benign_amap, cmap="jet", interpolation="nearest")
                    fig.colorbar(im, ax=ax, shrink=0.6, aspect=aspect, pad=0.02)
                    ax.set_title(f"Log10 Score: {anomaly_score:.2f}", fontweight="bold")

            else:  # Attack columns
                idx = cid - 1
                if rid == 0:
                    ax.imshow(adv_images[idx], cmap="gray")
                    ax.set_title(f"{label}+{attack_labels[idx]}", fontweight="bold")
                elif rid == 1:
                    anomaly_score = adv_amaps[idx].max()
                    im = ax.imshow(adv_amaps[idx], cmap="jet", interpolation="nearest")
                    cbar = fig.colorbar(im, ax=ax, shrink=0.6, aspect=aspect, pad=0.02)
                    ax.set_title(f"Log10 Score: {anomaly_score:.2f}", fontweight="bold")
                elif rid == 2:
                    ax.imshow(masked_adv_images[idx], cmap="gray")
                    ax.set_title(
                        f"{label}+Masked {attack_labels[idx]}", fontweight="bold"
                    )
                else:  # rid==3
                    anomaly_score = masked_adv_amaps[idx].max()
                    im = ax.imshow(
                        masked_adv_amaps[idx], cmap="jet", interpolation="nearest"
                    )
                    fig.colorbar(im, ax=ax, shrink=0.6, aspect=aspect, pad=0.02)
                    ax.set_title(f"Log10 Score: {anomaly_score:.2f}", fontweight="bold")

    # plt.subplots_adjust(right=0.88, wspace=0.3)  # Space for colorbars
    # plt.subplots_adjust(left=0.05, right=0.92, wspace=0.25, hspace=0.3)
    plt.tight_layout()
    plt.savefig(f"adv_amap_{atk_type}.pdf", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
label = "BENIGN"  # New label

# Find BENIGN directory
benign_path = None
for benign_path in Path(
    r"rosaid\results\evaluation\adversarial_evaluation\DNP3"
).iterdir():
    # Load images
    benign_img = cv2.imread(
        str(benign_path / f"{label}_clean.png"), cv2.IMREAD_GRAYSCALE
    )

    # Load anomaly maps
    data = np.load(benign_path / f"{label}_anomaly_maps.npz")
    benign_amap = data["clean_amap"]
    benign_amap = np.log10(benign_amap)  # Consistent with others

    # Create 1x2 side-by-side plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))

    # Image
    ax1.imshow(benign_img, cmap="gray")
    ax1.set_title("Benign Session Image", fontweight="bold")
    ax1.set_xticks([])
    ax1.set_yticks([])

    # Anomaly map with colorbar
    anomaly_score = benign_amap.max()
    im = ax2.imshow(benign_amap, cmap="jet", interpolation="nearest")
    cbar = fig.colorbar(im, ax=ax2, shrink=1.0, aspect=8, pad=0.02, fraction=0.046)
    ax2.set_title(
        f"Benign Anomaly Map\nLog10 Score: {anomaly_score:.2f}", fontweight="bold"
    )
    ax2.set_xticks([])
    ax2.set_yticks([])

    plt.tight_layout()
    plt.savefig(f"benign_{label}_comparison.pdf", dpi=300, bbox_inches="tight")
    plt.show()
    break


In [ ]:
data["anomaly_scores"]